In [1]:
import duckdb
import pandas as pd

In [2]:
con = duckdb.connect()

In [3]:
con.execute("INSTALL sqlite_scanner;")
con.execute("LOAD sqlite_scanner;")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

In [4]:
con.execute("ATTACH 'sakila.db' AS sakila (TYPE sqlite);")

In [5]:
con.execute("USE sakila;")
con.execute("SHOW TABLES;").df()

,name
0,actor
1,address
2,category
3,city
4,country
5,customer
6,customer_list
7,film
8,film_actor
9,film_category


In [6]:
query1 = """
SELECT 
    f.title AS film_title,
    c.name AS category_name,
    cu.first_name || ' ' || cu.last_name AS customer_name,
    ci.city, co.country,
    r.rental_date,
    (
        SELECT SUM(p.amount)
        FROM payment p
        JOIN rental r2 ON p.rental_id = r2.rental_id
        WHERE r2.customer_id = cu.customer_id
          AND r2.inventory_id IN (
              SELECT inventory_id FROM inventory WHERE film_id = f.film_id
          )
    ) AS total_paid
FROM rental r
JOIN inventory i ON r.inventory_id = i.inventory_id
JOIN film f ON i.film_id = f.film_id
JOIN film_category fc ON f.film_id = fc.film_id
JOIN category c ON fc.category_id = c.category_id
JOIN customer cu ON r.customer_id = cu.customer_id
JOIN address a ON cu.address_id = a.address_id
JOIN city ci ON a.city_id = ci.city_id
JOIN country co ON ci.country_id = co.country_id
LIMIT 10;
"""

df1 = con.execute(query1).df()
df1

,film_title,category_name,customer_name,city,country,rental_date,total_paid
0,WOMEN DORADO,Action,JORDAN ARCHULETA,Avellaneda,Argentina,2005-08-21 08:38:24,1.99
1,MOVIE SHAKESPEARE,Family,EVERETT BANDA,Bilbays,Egypt,2005-08-21 08:39:26,4.99
2,SPARTACUS CHEAPER,Family,NELLIE GARRETT,Shimoga,India,2005-08-21 08:40:21,8.99
3,FUGITIVE MAGUIRE,Travel,PETER MENARD,Ede,Netherlands,2005-08-21 08:40:56,4.99
4,GRAPES FURY,Foreign,WADE DELVALLE,Lausanne,Switzerland,2005-08-21 08:41:15,0.99
5,AMISTAD MIDSUMMER,New,DONNA THOMPSON,Elista,Russian Federation,2005-08-21 08:42:26,2.99
6,STEPMOM DREAM,Foreign,KRISTIN JOHNSTON,Sunnyvale,United States,2005-08-21 08:42:31,4.99
7,CRUELTY UNFORGIVEN,Classics,TYLER WREN,Rizhao,China,2005-08-21 08:54:26,0.99
8,AMADEUS HOLY,Action,SHERRY MARSHALL,Shubra al-Khayma,Egypt,2005-08-21 08:54:53,3.99
9,PRIMARY GLASS,Action,DELORES HANSEN,Jaroslavl,Russian Federation,2005-08-21 08:58:38,2.99


In [7]:
query_export = """
COPY (
    SELECT 
        f.title AS film_title, 
        c.name AS category_name,
        cu.first_name || ' ' || cu.last_name AS customer_name,
        ci.city, 
        co.country,
        SUM(p.amount) AS total_payment
    FROM payment p
    JOIN rental r ON p.rental_id = r.rental_id
    JOIN inventory i ON r.inventory_id = i.inventory_id
    JOIN film f ON i.film_id = f.film_id
    JOIN film_category fc ON f.film_id = fc.film_id
    JOIN category c ON fc.category_id = c.category_id
    JOIN customer cu ON r.customer_id = cu.customer_id
    JOIN address a ON cu.address_id = a.address_id
    JOIN city ci ON a.city_id = ci.city_id
    JOIN country co ON ci.country_id = co.country_id
    GROUP BY f.title, c.name, cu.customer_id, cu.first_name, cu.last_name, ci.city, co.country
) TO 'sakila_insights.csv' (HEADER, DELIMITER ',');
"""
con.execute(query_export)

In [8]:
pd.read_csv("sakila_insights.csv").head()

,film_title,category_name,customer_name,city,country,total_payment
0,GRADUATE LORD,Children,MANUEL MURRELL,Jaffna,Sri Lanka,6.98
1,GAMES BOWFINGER,Travel,DEANNA BYRD,Tuguegarao,Philippines,4.99
2,WHALE BIKINI,Foreign,BARRY LOVELACE,Kitwe,Zambia,4.99
3,REDS POCUS,Music,JUAN FRALEY,Teboksary,Russian Federation,4.99
4,THIEF PELICAN,Animation,CONSTANCE REID,Zaria,Nigeria,4.99


In [9]:
query_csv = """
SELECT film_title, SUM(total_payment) AS revenue
FROM 'sakila_insights.csv'
GROUP BY film_title
ORDER BY revenue DESC
LIMIT 5;
"""
con.execute(query_csv).df()

,film_title,revenue
0,TELEGRAPH VOYAGE,231.73
1,WIFE TURN,223.69
2,ZORRO ARK,214.69
3,GOODFELLAS SALUTE,209.69
4,SATURDAY LAMBS,204.72
